# Code Robot on Colab (TPU) - Resume Training

Click-through notebook to resume exactly where local work left off:
- `crawl_manifest.json` (40%+ of the site crawl) and the trained checkpoints are committed to the repo, so cloning/pulling here gets everything.
- Live web training now runs the `web` tier by default (fixed context-length rejection bug) or `two_billion_1m` on this TPU.
- Windowed attention, causal masking, data labeling, replay buffer, and `--stop`/resume are all already in the pushed code - nothing to re-apply.

**Before running**: Runtime -> Change runtime type -> select a TPU runtime.


### Step 1: Clone/update the repository
Pulls the latest code plus the committed `crawl_manifest.json` and `checkpoints/*.pt`.

In [ ]:
import os
REPO_DIR = '/content/robot'
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin main
else:
    !git clone https://github.com/nadeemmurtaza/robot.git {REPO_DIR}
    %cd {REPO_DIR}


### Step 2: Install dependencies
PyTorch + torch_xla for the TPU, Playwright (headless Chromium) for the web crawler, plus project deps.

If the `torch_xla` install line fails, check Colab's current TPU setup docs - the install URL changes between PyTorch/XLA releases.

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q "torch_xla[tpu]" -f https://storage.googleapis.com/libtpu-releases/index.html
!pip install -q playwright beautifulsoup4
!playwright install --with-deps chromium


### Step 3: Verify the TPU is visible

In [ ]:
import torch
try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    print("TPU device:", device)
except Exception as e:
    print("torch_xla / TPU not available:", e)
    print("CUDA available:", torch.cuda.is_available())


### Step 4: Inspect model tiers
Confirms the 2.04B / 1M-context tier (and everyone else) builds correctly before spending compute on it.

In [ ]:
!python robot.py --tiers

### Step 5: Resume the website crawl + live training
Auto-resumes from `crawl_manifest.json` (no `--fresh`, so it continues from ~40% rather than restarting).
`SITE_URL` below only matters if the manifest is missing/empty - while resuming, the manifest's own URLs are used, so leaving it as-is is fine.

Pick the tier for this run:
- `--tier web` - realistic on CPU/GPU, fixed the 64-byte too-long-rejection bug
- `--tier two_billion_1m` - the frontier tier, needs this notebook's TPU


In [ ]:
SITE_URL = "https://example.com"  # change only if crawl_manifest.json is missing/empty
!python browse.py "{SITE_URL}" --spider --train --tier two_billion_1m


### Step 5b: Stop the crawl gracefully
Run this in a *separate* cell while Step 5 is executing (or just hit Colab's Stop button on that cell - `browse.py` catches the interrupt and saves cleanly either way). Progress is saved after every page, so re-running Step 5 always resumes from exactly where this left off.

In [ ]:
!python browse.py --stop --manifest crawl_manifest.json

### Step 6 (optional): Train VQ-VAE + NCM generator directly at the 2B/1M tier
Separate from live web training - trains on the small built-in code corpus instead of scraped pages.

In [ ]:
!python -m custom_ncm.train_vqvae --tier two_billion_1m

In [ ]:
!python -m custom_ncm.train_ncm --tier two_billion_1m

### Step 7 (optional): Train on the official Python stdlib corpus at the 2B/1M tier
Runs the held-out correctness eval (syntax-valid % / exact-match %) at the end.

In [ ]:
!python train_on_official_python.py --tier two_billion_1m

### Step 8: Interactive generation
Sanity-check the model at whichever tier you've been training.

In [ ]:
!python robot.py --tier two_billion_1m --prompt "# Filter even numbers from list\ndef filter_evens(nums):\n"


### Step 9 (optional): Mount Google Drive and persist checkpoints there too
Uncomment if you want a second copy outside the repo/Colab runtime.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/robot_checkpoints
# !cp -r checkpoints/* /content/drive/MyDrive/robot_checkpoints/
# print("Checkpoints saved to Google Drive")


### Step 10: Push progress back to GitHub
So the next resume - here, locally, or on another machine - picks up from this point. Needs git credentials configured in this Colab session first (e.g. a GitHub personal access token via `!git config` / embedding it in the remote URL) - not included here since it's a secret.

In [ ]:
!git add -f checkpoints/*.pt checkpoints/replay_buffer.jsonl crawl_manifest.json
!git commit -m "chore: checkpoint progress from Colab run"
!git push origin main
